# IPMSM PyAEDT GUI Test Notebook

This notebook is intentionally short and uses the same modules as the final batch runner.

Run cells from top to bottom for a setup-only smoke test. Set `RUN_ANALYSIS_IN_NOTEBOOK = True` in the options cell when you want to watch the transient solve in the AEDT GUI.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

WORKSPACE = Path(r"Y:\git\pyaedt_motor")
if WORKSPACE.exists():
    os.chdir(WORKSPACE)
else:
    WORKSPACE = Path.cwd()

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from run_ipmsm_batch import (
    Simulation,
    build_spec,
    dataframe_first_row,
    export_ppt_reports,
    summarize_transient_outputs,
)
from module.ipmsm_geometry import create_ipmsm_design
import module.ipmsm_ppt_setup as ipmsm_ppt_setup
from pyaedt_module.core import pyDesktop

ipmsm_ppt_setup = importlib.reload(ipmsm_ppt_setup)
configure_ipmsm_from_ppt = ipmsm_ppt_setup.configure_ipmsm_from_ppt

WORKSPACE


In [ ]:
# Notebook options
SHOW_AEDT_GUI = True
RUN_ANALYSIS_IN_NOTEBOOK = False  # Change to True only when you want to run the transient solve.
NUM_CORES = 4

CASE = {
    "case_id": "notebook_test_10cycle",
    "pole_number": 8,
    "slot_number": 12,
    "symmetry_factor": 4,
    "base_rpm": 1200,
    "i_peak_a": 137.8,
    "beta_deg": 0,
    "series_turns_per_phase": 48,
    "turns_per_coil_side": 12,
    "stack_length_mm": 49.45,
    "phase_resistance_ohm": 0.01,
    "vdc_v": 200,
    "initial_position_deg": -22.5,
    "transient_periods": 10,
    "steps_per_period": 90,
}

NON_GRAPHICAL = not SHOW_AEDT_GUI
SIMULATION_DIR = WORKSPACE / "simulation"
SIMULATION_DIR.mkdir(parents=True, exist_ok=True)

CASE


In [ ]:
# Start AEDT and create a fresh project.
# close_on_exit=False keeps the GUI open so you can inspect progress/results.
desktop = pyDesktop(
    version=None,
    non_graphical=NON_GRAPHICAL,
    close_on_exit=False,
    new_desktop=True,
)

sim1 = Simulation(desktop=desktop, cores=NUM_CORES)
sim1.create_simulation_name(SIMULATION_DIR)
project1 = sim1.create_project(SIMULATION_DIR)
project_path = Path(project1.path)

{
    "simulation_name": sim1.PROJECT_NAME,
    "project_path": str(project_path),
    "gui_visible": SHOW_AEDT_GUI,
}


In [ ]:
# Build the full 360-degree IPMSM geometry.
design1, input_data, object_groups = create_ipmsm_design(project1, sim1)

{
    "design": "IPMSM",
    "objects": {key: len(value) for key, value in object_groups.items()},
    "input_preview": dataframe_first_row(input_data),
}


In [ ]:
# Apply materials, boundaries, windings, mesh, reports, and 10-cycle transient setup.
ppt_spec = build_spec(CASE, default_symmetry_factor=CASE["symmetry_factor"])

ppt_setup_result = configure_ipmsm_from_ppt(
    design1,
    object_groups=object_groups,
    spec=ppt_spec,
    operation="sin_current",
    use_periodic_boundary=False,  # Full 360 model: keep False.
    create_missing_region=True,
    create_missing_band=True,
    create_reports=True,
    clear_existing=True,
    analyze=False,  # Keep setup and solve separated for notebook debugging.
    cores=NUM_CORES,
)

ppt_setup_result


In [ ]:
# Optional visible solve.
# If RUN_ANALYSIS_IN_NOTEBOOK=True, watch the AEDT progress window while this cell runs.
if RUN_ANALYSIS_IN_NOTEBOOK:
    m2d = getattr(design1, "solver_instance", design1)
    notebook_analysis_result = m2d.analyze(
        setup=ppt_spec.setup_name,
        cores=NUM_CORES,
        use_auto_settings=False,
    )
    if notebook_analysis_result is False:
        print("PyAEDT returned False. AEDT may still have solved data, so run the export cell to check reports.")
else:
    notebook_analysis_result = "Skipped. Set RUN_ANALYSIS_IN_NOTEBOOK = True in the options cell to solve."

notebook_analysis_result


In [ ]:
# Export reports and summarize outputs.
# This cell returns an empty output summary if the solve was skipped or reports have no solved data yet.
notebook_exported_reports = export_ppt_reports(design1, project_path, CASE["case_id"])
notebook_output_summary = summarize_transient_outputs(notebook_exported_reports, ppt_spec)

{
    "exported_reports": notebook_exported_reports,
    "output_summary": notebook_output_summary,
}


In [ ]:
# Optional cleanup cell. Run manually when you are done inspecting AEDT.
# desktop.release_desktop(close_projects=True, close_on_exit=True)
# "AEDT desktop released"
